In [1]:
import java.time.LocalTime
import java.time.format.DateTimeFormatter
import kotlin.text.MatchResult
import kotlin.text.get

// ========== Domain model ==========
sealed class LogEvent {
    data class ConfirmationRuleUpdate(
        val slot: Long,
        val headSlot: Long,
        val headShortRootHex: String,           // e.g., 0x9acceb
        val confirmedSlot: Long,
        val confirmedShortRootHex: String,      // e.g., 0x6dba83
        val justifiedEpoch: Long
    ) : LogEvent()

    data class SlotEvent(
        val time: LocalTime,
        val slot: Long,
        val blockRootHex: String?,              // null when "... empty"
    ) : LogEvent()
    
    data class NewVotestInBlock(
        val slot: Long,
        val newVotes: Int
    ) : LogEvent()

    data class SlotBlockVotes(
        val slot: Long,
        val blockVotes: Int,
        val slotVotes: Int
    ) : LogEvent()

    data class Unknown(val line: String) : LogEvent()
}

// ========== Parser ==========
object TekuLogParser {
    private val timeFmt = DateTimeFormatter.ofPattern("HH:mm:ss.SSS")

    // 1) updateConfirmationRuleStore
    private val reUpdate = Regex(
        """
        ^updateConfirmationRuleStore:\s*
        slot=(?<slot>\d+),\s*
        head=(?<head>\d+),\(
            (?<headHex>0x[0-9a-fA-F]+)
        \),\s*
        confirmed=(?<conf>\d+)\(
            (?<delta>-?\d+)
        \),\(
            (?<confHex>0x[0-9a-fA-F]+)
        \),\s*
        states\s+requested/uniq:\s*
            (?<req>\d+)/
            (?<uniq>\d+),\s*
        justified=Checkpoint\[
            (?<jEpoch>\d+),\s*
            (?<jHex>0x[0-9a-fA-F]+)
        \]\s*in\s*
            (?<ms>\d+)\s*ms
        $
        """.trimIndent().compactWs(),
        setOf()
    )

    // 2a) Full Slot Event
    private val reSlotFull = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Slot\s+Event\s+\*{3}\s+
        Slot:\s*(?<slot>\d+),\s*
        Block:\s*(?<block>[0-9a-fA-F]{64}),\s*
        Justified:\s*(?<j>\d+),\s*
        Finalized:\s*(?<f>\d+),\s*
        Peers:\s*(?<peers>\d+)
        $
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    // 2b) Short/empty Slot Event (… empty and the rest may be ellipsis)
    private val reSlotEmpty = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Slot\s+Event\s+\*{3}\s+
        Slot:\s*(?<slot>\d+),\s*
        Block:\s*\.{3}\s*empty
        (?:,.*)?$
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    private val newVotesInBlock = Regex(
        """
        ^Importing\sblock:\s(?<slot>\d+),.+\svotes\sin\sblock:\s(?<votes>\d+)\s*$
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    private val slotBlockVotes = Regex(
        """
        ^\tSlot/block\svotes/slot\svotes:\t(?<slot>\d+)\t(?<blockVotes>\d+)\t(?<slotVotes>\d+)$
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    
    fun parse(line: String): LogEvent {
        reUpdate.matchEntire(line)?.let { m ->
            return LogEvent.ConfirmationRuleUpdate(
                slot = m.group("slot").toLong(),
                headSlot = m.group("head").toLong(),
                headShortRootHex = m.group("headHex"),
                confirmedSlot = m.group("conf").toLong(),
                confirmedShortRootHex = m.group("confHex"),
                justifiedEpoch = m.group("jEpoch").toLong(),
            )
        }

        reSlotFull.matchEntire(line)?.let { m ->
            return LogEvent.SlotEvent(
                time = LocalTime.parse(m.group("time"), timeFmt),
                slot = m.group("slot").toLong(),
                blockRootHex = m.group("block")
            )
        }

        reSlotEmpty.matchEntire(line)?.let { m ->
            return LogEvent.SlotEvent(
                time = LocalTime.parse(m.group("time"), timeFmt),
                slot = m.group("slot").toLong(),
                blockRootHex = null,            // ... empty
            )
        }

        newVotesInBlock.matchEntire(line)?.let { m ->
            return LogEvent.NewVotestInBlock(
                slot = m.group("slot").toLong(),
                newVotes = m.group("votes").toInt()
            )
        }

        slotBlockVotes.matchEntire(line)?.let { m ->
            return LogEvent.SlotBlockVotes(
                slot = m.group("slot").toLong(),
                blockVotes = m.group("blockVotes").toInt(),
                slotVotes = m.group("slotVotes").toInt()
            )
        }

        return LogEvent.Unknown(line)
    }

    // --- helpers ---

    private fun MatchGroupCollection.getByName(name: String) =
        (this as MatchNamedGroupCollection).get(name)

    private fun MatchResult.group(name: String): String =
        this.groups.getByName(name)?.value ?: error("Missing group '$name'")

    // Tolerate varied whitespace without making the regex unreadable
    private fun String.compactWs(): String =
        replace("\n", "").replace(Regex("\\s+"), "\\s*")

    // "arrival 8760ms, gossip_validation +0ms, processed +85ms, ..."
    private fun parseTimings(raw: String): LinkedHashMap<String, Long> {
        val map = LinkedHashMap<String, Long>()
        val pair = Regex("""([a-zA-Z0-9_]+)\s+([+\-]?\d+)ms""")
        for (part in raw.split(Regex("""\s*,\s*"""))) {
            val m = pair.find(part) ?: continue
            val key = m.groupValues[1]
            val v = m.groupValues[2].toLong()
            map[key] = v
        }
        return map
    }
}

// ========== Quick demo ==========
val lines1 = listOf(
    "Importing block: 12657893, 0x82929bc039a20ce96ba5c4658fe4e289f72c980778966300c1d256276e96663d, votes in block: 66",
    "	Slot/block votes/slot votes:	13181949	30792	30796",
    "New votes from prev slot: 13255299, 30804",
    "updateConfirmationRuleStore: slot=12657889, head=12657889,(0x929850), confirmed=12657888(-1),(0x3b1398), states requested/uniq: 2/1, justified=Checkpoint[395559, 0x3b1398] in 1 ms",
    // old log line
    "updateConfirmationRuleStore: head=12657983,(0x9acceb), confirmed=12657982(-1),(0x6dba83), states requested/uniq: 5/2, justified=Checkpoint[395560, 0xe30323] in 598 ms",
    "17:17:03.812 INFO  - Slot Event  *** Slot: 12657983, Block: 9accebd69baa3e90411f123121f9eb7d2fc61f8117d81accf87f81c7bafde347, Justified: 395560, Finalized: 395559, Peers: 63",
    "17:05:51.527 INFO  - Slot Event  *** Slot: 12657927, Block: ... empty,    Justified: ...",
    "17:17:11.002 INFO  - Epoch Event *** Epoch: 395562, Justified checkpoint: 395561, Finalized checkpoint: 395560, Finalized root: e3032378ff11e040a689579c8a3eefa4e45485ee2ff270709a431106aac568a7",
    "18:23:07.855 WARN  - Late Block Import *** Block: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313) Proposer: 3126 Result: success Timings: arrival 8760ms, gossip_validation +0ms, pre-state_retrieved +2ms, processed +85ms, data_availability_checked +0ms, execution_payload_result_received +0ms, begin_importing +0ms, transaction_prepared +0ms, transaction_committed +0ms, completed +8ms",
    "18:23:13.482 INFO  - Reorg Event *** New Head: 6804d44b51fd3b957c147298575f53e8929769dfba4b6d80ca4725a08b50176d (12658314), Previous Head: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313), Common Ancestor: 41b24017534a5266d112fea39e7f7e10c1fade19272ee7e427b42cd586e2853a (12658312)"
)
val parsed = lines1.map(TekuLogParser::parse)

parsed.joinToString("\n")


NewVotestInBlock(slot=12657893, newVotes=66)
SlotBlockVotes(slot=13181949, blockVotes=30792, slotVotes=30796)
ConfirmationRuleUpdate(slot=12657889, headSlot=12657889, headShortRootHex=0x929850, confirmedSlot=12657888, confirmedShortRootHex=0x3b1398, justifiedEpoch=395559)
Unknown(line=updateConfirmationRuleStore: head=12657983,(0x9acceb), confirmed=12657982(-1),(0x6dba83), states requested/uniq: 5/2, justified=Checkpoint[395560, 0xe30323] in 598 ms)
SlotEvent(time=17:17:03.812, slot=12657983, blockRootHex=9accebd69baa3e90411f123121f9eb7d2fc61f8117d81accf87f81c7bafde347)
SlotEvent(time=17:05:51.527, slot=12657927, blockRootHex=null)
Unknown(line=17:17:11.002 INFO  - Epoch Event *** Epoch: 395562, Justified checkpoint: 395561, Finalized checkpoint: 395560, Finalized root: e3032378ff11e040a689579c8a3eefa4e45485ee2ff270709a431106aac568a7)
Unknown(line=18:23:07.855 WARN  - Late Block Import *** Block: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313) Proposer: 3126

In [2]:
%use dataframe

In [10]:
import java.io.File

class ConfLogs(
    val logEvents: List<LogEvent>

) {

    constructor(logFile: String) : this(
        File(logFile)
        .readLines()
        .map { TekuLogParser.parse(it) }
        .filter { it !is LogEvent.Unknown }
    )

    val confirmLag: List<Pair<Long, Long>> = logEvents
        .filterIsInstance<LogEvent.ConfirmationRuleUpdate>()
        .map { it.slot to (it.slot - it.confirmedSlot) }

    val confLagDF = confirmLag.toDataFrame()
        .rename { all() }.into("slot", "confirm_lag")
        .inferType()

    val justifyLag = logEvents
        .filterIsInstance<LogEvent.ConfirmationRuleUpdate>()
        .map { it.slot to (it.slot - it.justifiedEpoch * 32) }

    val justifyLagDF = justifyLag.toDataFrame()
        .rename { all() }.into("slot", "justify_lag")
        .inferType()

    val slotVotes: Map<Long, Int> = logEvents
        .filterIsInstance<LogEvent.SlotBlockVotes>()
        .map { it.slot to it.slotVotes }
        .groupingBy { it.first }
        .fold(0) { a, e -> max(a, e.second) }
}


fun List<ConfLogs>.mergeSimple(): ConfLogs = ConfLogs(this.flatMap { it.logEvents })


//val runs = mapOf(
//    "1. noOpts" to ConfLogs("./conf-sim-7-no-opt.log"),
//    "2. emptySlotOptOnly" to ConfLogs("./conf-sim-7-empty-slot-opt.log"),
//    "3. emptySlotAndLateBlockOpt" to ConfLogs("./conf-sim-6-late-block-opt.log"),
//)

val logs3 = ConfLogs("/Users/nashatyrev/IdeaProjects/teku/ethereum/statetransition/replayWithXatuAttestations-3.log")
val logs3_1 = ConfLogs("/Users/nashatyrev/IdeaProjects/teku/ethereum/statetransition/replayWithXatuAttestations-3.1.log")
val logs3_2 = ConfLogs("/Users/nashatyrev/IdeaProjects/teku/ethereum/statetransition/replayWithXatuAttestations-3.2.log")
val logs4 = ConfLogs("/Users/nashatyrev/IdeaProjects/teku/ethereum/statetransition/replayWithXatuAttestations-4.log")

val logs0 = listOf(logs3, logs3_1, logs3_2, logs4).mergeSimple()


val runs = mapOf(
    "0" to logs0
)

val runsDf = runs
    .map { (name, logs) ->
        logs.confLagDF.add("run") { name }
    }
    .reduce { a1, a2 -> a1.concat(a2) }

In [7]:
%use kandy


In [8]:
val justLagDf = logs0.justifyLagDF

In [22]:
justLagDf
    .filter {
        it.slot > 13_165_000 && it.slot < 13_166_800
    }
    .plot {
        x(slot)
        y(justify_lag)
        bars {
            this.borderLine {
                this.type = LineType.BLANK
                this.width = 0.0
            }
            fillColor = Color.BLUE

        }

        layout {
            size = 2500 to 2000
            style {
                panel.grid.majorXLine { blank = true }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="dFnzCS"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2500.0, 
 height: 2000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("dFnzCS");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"justify_lag":[41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,3

In [5]:
val slotVotesDf = logs0.slotVotes.entries
    .toDataFrame()
    .rename { all() }.into("slot", "slot_votes")
    .inferType()

In [9]:
slotVotesDf
    .filter {
        it.slot > 13_165_800 && it.slot < 13_168_100
//        it.slot > 13_166_000 && it.slot < 13_166_500
    }
    .plot {
        x(slot)
        y(slot_votes)
        bars {
            this.borderLine {
                this.type = LineType.BLANK
                this.width = 0.0
            }
            fillColor = Color.BLUE

        }

        layout {
            size = 2500 to 2000
            style {
                panel.grid.majorXLine { blank = true }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ygFC9f"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2500.0, 
 height: 2000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("ygFC9f");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"slot_votes":[30370.0,30402.0,30363.0,30345.0,30387.0,30378.0,30334.0,30388.0,30391.0,30306.0,30365.0,30386.0,30373.0,30355.0,30372.0,30353.0,30290.0,30375.0,30398.0,30401.0,30386.0,30401.0,30334.0,30139.0,29881.0,30229.0,30216.0,30311.0,30215.0,30048.0,29871.0,30321.0,30248.0,30426.0,30387.0,30434.0,30417.0,30462.0,30460.0,30453.0,30426.0,30488.0,30490.0,30432.0,30364.0,30394.0,30417.0,30392.0,30477.0,30485.0,30387.0,29991.0,30455.0,30445.0,30379.0,29874.0,29999.0,30169.0,30276.0,30256.0,30309.0,30133.0,30257.0,30339.0,30367.0,30329.0,30352.0,30344.0,30386.0,30349.0,30363.0,30336.0,30352.0,30024.0,30259.0,30363.0,30234.0,29828.0,30165.0,30375.0,30423.0,30388.0,30389.0,30403.0,30424.0,30423.0,30430.0,30282.0,30079.0,30208.0,30267.0,30319.0,30392.0,30380.0,30260.0,30336.0,30377.0,30420.0,30254.0,30464.0,30485.0,30438.0,30479.0,30547.0,30501.0,30521.0,30529.0,30520.0,30571.0,30446.0,30531.0,30532.0,30515.0,30530.0,30516.0,30384.0,30392.0,30394.0,30536.0,30387.0,30297.0,30353.0,30387.0,30358.0,30432.0,30426.0,30374.0,30417.0,30394.0,30377.0,30377.0,30414.0,30389.0,30436.0,30403.0,30425.0,30400.0,30402.0,30262.0,30396.0,30413.0,30398.0,30388.0,30340.0,30351.0,30436.0,30363.0,30413.0,30406.0,30243.0,30169.0,29925.0,29485.0,29845.0,30191.0,30217.0,30367.0,30367.0,30321.0,30319.0,30418.0,30406.0,30457.0,30398.0,30427.0,30435.0,30408.0,30480.0,30435.0,30438.0,30434.0,30185.0,30429.0,30448.0,30452.0,30464.0,30468.0,30487.0,30368.0,30236.0,30150.0,30479.0,30431.0,30240.0,30064.0,30080.0,30156.0,30072.0,30215.0,30300.0,30276.0,30283.0,30323.0,30277.0,30231.0,30352.0,30305.0,30337.0,30303.0,30270.0,30305.0,30278.0,30291.0,30311.0,30303.0,30337.0,30300.0,30407.0,30373.0,30347.0,30380.0,30370.0,30413.0,30384.0,30103.0,29888.0,29325.0,27945.0,29337.0,29602.0,29782.0,30031.0,30160.0,29972.0,30169.0,29490.0,29562.0,29729.0,30157.0,30054.0,30271.0,30098.0,30198.0,30035.0,30275.0,29514.0,29925.0,30060.0,30122.0,30141.0,29896.0,29615.0,29885.0,29998.0,29583.0,29704.0,29859.0,29477.0,26864.0,27214.0,27008.0,27914.0,26226.0,25388.0,25416.0,26037.0,27265.0,27535.0,26795.0,27106.0,27156.0,26582.0,26960.0,27992.0,28659.0,28328.0,28766.0,27887.0,0.0,26828.0,26620.0,27191.0,28294.0,27101.0,27838.0,28125.0,27604.0,28371.0,27492.0,26312.0,23570.0,23443.0,23757.0,23885.0,23749.0,23806.0,23713.0,23887.0,23805.0,24079.0,24078.0,23997.0,24152.0,24077.0,24491.0,24305.0,24476.0,24066.0,24596.0,24468.0,24457.0,24421.0,24601.0,24518.0,24406.0,24444.0,23877.0,24712.0,24721.0,24736.0,24628.0,23758.0,23301.0,23306.0,23391.0,23432.0,23498.0,23615.0,23658.0,23564.0,23693.0,23515.0,23841.0,23775.0,23532.0,23300.0,23545.0,23541.0,23379.0,23248.0,23732.0,23391.0,23384.0,23592.0,23560.0,23867.0,22940.0,23754.0,23626.0,23555.0,23742.0,23890.0,23536.0,23394.0,23383.0,23280.0,23339.0,22869.0,23307.0,23450.0,22623.0,22974.0,23287.0,23695.0,23512.0,23501.0,23628.0,23616.0,23791.0,23704.0,23871.0,23547.0,23818.0,23977.0,24009.0,24272.0,24109.0,24209.0,24068.0,24112.0,24072.0,24070.0,24085.0,23770.0,23909.0,23320.0,23670.0,23552.0,23265.0,2219

In [52]:
import org.jetbrains.kotlinx.kandy.letsplot.layers.builders.subcontext.BorderLine


runsDf
    .filter {
        it.slot > 13_165_800 && it.slot < 13_167_600
    }
    .plot {
        x(slot)
        y(confirm_lag)
        bars {
            this.borderLine {
                this.type = LineType.BLANK
                this.width = 0.0
            }
            fillColor = Color.BLUE

        }

        facetWrap(nCol = 1) {
            facet(run)
        }

        layout {
            size = 2500 to 2000
            style {
                panel.grid.majorXLine { blank = true }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="oPPbaL"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2500.0, 
 height: 2000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("oPPbaL");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"run":["0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0","0"

In [ ]:
runs.map { (name, v) -> 
    val lags = v.confirmLag.map { it.second }
    val countMap = lags.groupingBy { it }.eachCount().toSortedMap()
    name + ":\n"+ countMap
}.joinToString("\n")    

In [ ]:
val logs = runs.entries.first().value
val updEvents = logs.logEvents
    .filterIsInstance<LogEvent.ConfirmationRuleUpdate>()
val blockCount = updEvents
    .map { it.headSlot }
    .distinct()
    .count()
val slotCount = updEvents.last().slot - updEvents.first().slot

"Empty slots count: " + (slotCount - blockCount)

In [13]:

val lags = df1.getColumn { confirm_lag }.toList()
val map = lags.groupingBy { it }.eachCount().toSortedMap()

val totCnt = map.values.sum()
val percentMap = map.mapValues { it.value.toDouble() * 100 / totCnt }

val dfMap = mapOf(
    "conf_distance" to percentMap.keys.toList(),
    "fraction" to percentMap.values.toList(),
)

plot(dfMap) {
    bars {
        x(percentMap.keys) {
            axis {
                name = "Confirmation distance in slots"
            }
        }
        y(percentMap.values) {
            axis {
                name = "Fraction in %%"
            }
        }
    }
}


org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[13], line 2, column 12: Unresolved reference: df1
at Cell In[13], line 2, column 28: Unresolved reference. None of the following candidates is applicable because of receiver type mismatch: 
public final val ColumnsScope<Line_14_jupyter._DataFrameType>.confirm_lag: DataColumn<Long> defined in Line_14_jupyter
public final val DataRow<Line_14_jupyter._DataFrameType>.confirm_lag: Long defined in Line_14_jupyter
at Cell In[13], line 3, column 29: Unresolved reference: it
at Cell In[13], line 6, column 34: Unresolved reference: it
at Cell In[13], line 8, column 13: Not enough information to infer type variable V

In [ ]:
map

In [ ]:
import org.jetbrains.letsPlot.core.spec.back.transform.bistro.util.scale

plot { 
    histogram(x = df1.getColumn { confirm_lag }) {
        y { 
            scale = continuous(transform = Transformation.LOG2)
        }
    } 
}

In [ ]:
val newVotesInBlock = logEvents
    .filterIsInstance<LogEvent.NewVotestInBlock>()
    .map { it.newVotes }
    .filter { it < 2000 }

plot {
    histogram(newVotesInBlock) {
//        y {
//            scale = continuous(transform = Transformation.LOG2)
//        }
    }
}

In [ ]:
"" + newVotesInBlock.sum() + " in " + newVotesInBlock.size + " blocks, average: " + (newVotesInBlock.sum() / newVotesInBlock.size)

## Large data analysis (order of > 10 days)

In [11]:
val confLagDF = logs0.confLagDF.inferType()

In [53]:
val df10 = confLagDF
    .add("epoch") { slot / 32 / 100 * 100 }

In [54]:
val stats = df10.groupBy { epoch }
    .aggregate {
        median { confirm_lag } into "p50"
        percentile(90.0) { confirm_lag } into "p90"
        percentile(95.0) { confirm_lag } into "p95"
        percentile(99.0) { confirm_lag } into "p99"
        max { confirm_lag } into "max"
    }

In [55]:
stats.plot {
    boxes {
        x(epoch)
        yMin(p50)
        lower(p90)
        middle(p95)
        upper(p99)
        yMax(max)
    }
    layout {
        size = 2500 to 2000

    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="bVQVQM"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2500.0, 
 height: 2000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("bVQVQM");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"p99":[3.0,3.0,3.0,3.0,3.0,87.98333333333358,3.0,3.0,3.0,3.0,3.0,3.0,3.0,2.0,6.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,2.663333333333412,3.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,6.0,6.0,6.0,6.0,7.0,6.0,7.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,7.0,7.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0],
"max":[7.0,5.0,6.0,7.0,7.0,95.0,7.0,7.0,7.0,7.0,4.0,4.0,4.0,3.0,8.0,4.0,6.0,7.0,5.0,6.0,5.0,4.0,7.0,3.0,4.0,3.0,5.0,3.0,3.0,3.0,3.0,4.0,3.0,8.0,7.0,7.0,7.0,7.0,7.0,8.0,8.0,7.0,7.0,7.0,7.0,8.0,7.0,7.0,8.0,7.0,7.0,10.0,7.0,8.0,7.0,8.0,8.0,7.0,7.0,7.0,8.0,7.0,7.0,8.0,7.0,8.0,7.0,7.0,7.0,9.0,7.0,8.0,10.0,7.0,7.0,8.0,7.0,7.0],
"p90":[1.0,1.0,1.0,1.0,1.0,32.0,2.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,5.0,5.0,6.0,6.0,6.0,6.0,6.0,6.0,5.0,6.0,6.0,5.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,5.0,6.0,5.0,5.0,5.0,6.0,5.0,5.0,5.0,6.0,6.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0],
"epoch":[410900.0,411000.0,411100.0,411200.0,411300.0,411400.0,411500.0,411600.0,411700.0,411800.0,411900.0,412000.0,412100.0,412200.0,412300.0,412400.0,412500.0,412600.0,412700.0,412800.0,412900.0,413000.0,413100.0,413200.0,413300.0,413400.0,413500.0,413600.0,413700.0,413800.0,413900.0,414000.0,414100.0,414200.0,414300.0,414400.0,414500.0,414600.0,414700.0,414800.0,414900.0,415000.0,415100.0,415200.0,415300.0,415400.0,415500.0,415600.0,415700.0,415800.0,415900.0,416000.0,416100.0,416200.0,416300.0,416400.0,416500.0,416600.0,416700.0,416800.0,416900.0,417000.0,417100.0,417200.0,417300.0,417400.0,417500.0,417600.0,417700.0,417800.0,417900.0,418000.0,418100.0,418200.0,418300.0,418400.0,418500.0,418600.0],
"p50":[1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0],
"p95":[2.0,2.0,1.0,2.0,2.0,48.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,1.0,6.0,1.0,1.0,2.0,2.0,2.0,1.650000000000091,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0]
},
"ggsize":{
"width":2500.0,
"height":2000.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"epoch",
"ymin":"p50",
"lower":"p90",
"middle":"p95",
"upper":"p99",
"ymax":"max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"boxplot",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"epoch"
},{
"type":"float",
"column":"p50"
},{
"type":"float",
"column":"p90"
},{
"type":"float",
"column":"p95"
},{
"type":"float",
"column":"p99"
},{
"type":"int",
"column":"max"
}]
},
"spec_id":"44"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizin

In [56]:
confLagDF
    .filter { (slot / 32 / 100 * 100) == 412300L }
    .plot {
        line {
            x(slot)
            y(confirm_lag)
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="JBvn9V"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("JBvn9V");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"slot":[1.31936E7,1.3193601E7,1.3193602E7,1.3193603E7,1.3193604E7,1.3193605E7,1.3193606E7,1.3193607E7,1.3193608E7,1.3193609E7,1.319361E7,1.3193611E7,1.3193612E7,1.3193613E7,1.3193614E7,1.3193615E7,1.3193616E7,1.3193617E7,1.3193618E7,1.3193619E7,1.319362E7,1.3193621E7,1.3193622E7,1.3193623E7,1.3193624E7,1.3193625E7,1.3193626E7,1.3193627E7,1.3193628E7,1.3193629E7,1.319363E7,1.3193631E7,1.3193632E7,1.3193633E7,1.3193634E7,1.3193635E7,1.3193636E7,1.3193637E7,1.3193638E7,1.3193639E7,1.319364E7,1.3193641E7,1.3193642E7,1.3193643E7,1.3193644E7,1.3193645E7,1.3193646E7,1.3193647E7,1.3193648E7,1.3193649E7,1.319365E7,1.3193651E7,1.3193652E7,1.3193653E7,1.3193654E7,1.3193655E7,1.3193656E7,1.3193657E7,1.3193658E7,1.3193659E7,1.319366E7,1.3193661E7,1.3193662E7,1.3193663E7,1.3193664E7,1.3193665E7,1.3193666E7,1.3193667E7,1.3193668E7,1.3193669E7,1.319367E7,1.3193671E7,1.3193672E7,1.3193673E7,1.3193674E7,1.3193675E7,1.3193676E7,1.3193677E7,1.3193678E7,1.3193679E7,1.319368E7,1.3193681E7,1.3193682E7,1.3193683E7,1.3193684E7,1.3193685E7,1.3193686E7,1.3193687E7,1.3193688E7,1.3193689E7,1.319369E7,1.3193691E7,1.3193692E7,1.3193693E7,1.3193694E7,1.3193695E7,1.3193696E7,1.3193697E7,1.3193698E7,1.3193699E7,1.31937E7,1.3193701E7,1.3193702E7,1.3193703E7,1.3193704E7,1.3193705E7,1.3193706E7,1.3193707E7,1.3193708E7,1.3193709E7,1.319371E7,1.3193711E7,1.3193712E7,1.3193713E7,1.3193714E7,1.3193715E7,1.3193716E7,1.3193717E7,1.3193718E7,1.3193719E7,1.319372E7,1.3193721E7,1.3193722E7,1.3193723E7,1.3193724E7,1.3193725E7,1.3193726E7,1.3193727E7,1.3193728E7,1.3193729E7,1.319373E7,1.3193731E7,1.3193732E7,1.3193733E7,1.3193734E7,1.3193735E7,1.3193736E7,1.3193737E7,1.3193738E7,1.3193739E7,1.319374E7,1.3193741E7,1.3193742E7,1.3193743E7,1.3193744E7,1.3193745E7,1.3193746E7,1.3193747E7,1.3193748E7,1.3193749E7,1.319375E7,1.3193751E7,1.3193752E7,1.3193753E7,1.3193754E7,1.3193755E7,1.3193756E7,1.3193757E7,1.3193758E7,1.3193759E7,1.319376E7,1.3193761E7,1.3193762E7,1.3193763E7,1.3193764E7,1.3193765E7,1.3193766E7,1.3193767E7,1.3193768E7,1.3193769E7,1.319377E7,1.3193771E7,1.3193772E7,1.3193773E7,1.3193774E7,1.3193775E7,1.3193776E7,1.3193777E7,1.3193778E7,1.3193779E7,1.319378E7,1.3193781E7,1.3193782E7,1.3193783E7,1.3193784E7,1.3193785E7,1.3193786E7,1.3193787E7,1.3193788E7,1.3193789E7,1.319379E7,1.3193791E7,1.3193792E7,1.3193793E7,1.3193794E7,1.3193795E7,1.3193796E7,1.3193797E7,1.3193798E7,1.3193799E7,1.31938E7,1.3193801E7,1.3193802E7,1.3193803E7,1.3193804E7,1.3193805E7,1.3193806E7,1.3193807E7,1.3193808E7,1.3193809E7,1.319381E7,1.3193811E7,1.3193812E7,1.3193813E7,1.3193814E7,1.3193815E7,1.3193816E7,1.3193817E7,1.3193818E7,1.3193819E7,1.319382E7,1.3193821E7,1.3193822E7,1.3193823E7,1.3193824E7,1.3193825E7,1.3193826E7,1.3193827E7,1.3193828E7,1.3193829E7,1.319383E7,1.3193831E7,1.3193832E7,1.3193833E7,1.3193834E7,1.3193835E7,1.3193836E7,1.3193837E7,1.3193838E7,1.3193839E7,1.319384E7,1.3193841E7,1.3193842E7,1.3193843E7,1.3193844E7,1.3193845E7,1.3193846E7,1.3193847E7,1.3193848E7,1.3193849E7,1.319385E7,1.3193851E7,1.3193852E7,1.3193853E7,1.3193854E7,1.3193855

In [23]:
val slotVotesDf = logs0.slotVotes.entries
    .toDataFrame()
    .rename { all() }.into("slot", "slot_votes")
    .inferType()

In [57]:

slotVotesDf
    .filter {
//        it.slot > 13_254_000 && it.slot < 13_256_000
        it.slot > 13_193_000 && it.slot < 13_195_000
    }
    .plot {
        x(slot)
        y(slot_votes)
        bars {
            this.borderLine {
                this.type = LineType.BLANK
                this.width = 0.0
            }
            fillColor = Color.BLUE

        }

        layout {
            size = 2500 to 2000
            style {
                panel.grid.majorXLine { blank = true }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="oCHVwg"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2500.0, 
 height: 2000.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("oCHVwg");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"slot_votes":[31008.0,30991.0,31004.0,30997.0,31023.0,30952.0,30971.0,30975.0,31015.0,31000.0,31014.0,31006.0,31004.0,31010.0,31017.0,31028.0,31027.0,31026.0,31022.0,31015.0,31048.0,30986.0,31030.0,30872.0,30950.0,31009.0,30998.0,30971.0,31023.0,31013.0,31021.0,31027.0,31002.0,31020.0,31037.0,31017.0,31041.0,31037.0,31015.0,31014.0,31013.0,31028.0,31020.0,31025.0,31024.0,31025.0,31015.0,31005.0,31038.0,31024.0,31049.0,31036.0,31027.0,31043.0,31008.0,30303.0,30975.0,30931.0,31014.0,31017.0,31012.0,31018.0,31003.0,30890.0,31007.0,31007.0,31028.0,31010.0,31028.0,31004.0,31003.0,31037.0,31006.0,30991.0,31006.0,30992.0,31012.0,31017.0,30999.0,31004.0,30999.0,30996.0,31026.0,31031.0,30987.0,31008.0,31039.0,30978.0,31020.0,30974.0,30957.0,31022.0,31012.0,31028.0,30986.0,31027.0,31014.0,31005.0,31005.0,30983.0,31007.0,31018.0,31012.0,30986.0,30977.0,30979.0,31032.0,30979.0,30958.0,31004.0,31018.0,31034.0,30979.0,30990.0,31023.0,31037.0,31008.0,31055.0,31015.0,31037.0,31014.0,31016.0,31029.0,31020.0,30985.0,30987.0,31002.0,30917.0,30976.0,30962.0,30947.0,30997.0,31016.0,30943.0,30954.0,30844.0,31004.0,31034.0,31045.0,31023.0,30985.0,30948.0,30985.0,31033.0,30970.0,30948.0,30963.0,30989.0,30962.0,31018.0,31018.0,31018.0,31000.0,31028.0,30995.0,31008.0,30962.0,30926.0,30903.0,30948.0,30999.0,31004.0,30987.0,30979.0,30949.0,31025.0,31038.0,31014.0,30984.0,30984.0,31003.0,31043.0,31033.0,30984.0,31025.0,31019.0,30981.0,30976.0,31021.0,31011.0,31024.0,31038.0,31037.0,30952.0,30971.0,30988.0,31004.0,30982.0,30925.0,31041.0,30958.0,31037.0,30967.0,30969.0,30940.0,30951.0,30949.0,30956.0,31013.0,31025.0,31006.0,30990.0,30972.0,30937.0,30930.0,30929.0,31029.0,31028.0,31028.0,31002.0,31019.0,31027.0,30974.0,30935.0,31001.0,30941.0,30948.0,31009.0,31032.0,31018.0,30932.0,31008.0,31027.0,31031.0,30996.0,31025.0,31062.0,31041.0,31024.0,31025.0,31050.0,31056.0,31014.0,31041.0,31045.0,31016.0,31003.0,31043.0,31022.0,31032.0,31021.0,31031.0,31033.0,31032.0,31040.0,31052.0,31041.0,30974.0,30984.0,30998.0,31039.0,31004.0,30996.0,30971.0,31022.0,31024.0,31033.0,31022.0,31013.0,31026.0,30993.0,30958.0,31006.0,31010.0,31027.0,31021.0,31032.0,31022.0,31037.0,31024.0,31015.0,31023.0,30988.0,31028.0,31021.0,30969.0,31042.0,31027.0,31019.0,30961.0,31012.0,31033.0,31037.0,31029.0,31035.0,31032.0,31005.0,30959.0,31004.0,31029.0,30995.0,31010.0,30949.0,31028.0,30994.0,31018.0,31043.0,31022.0,31059.0,30993.0,31012.0,31051.0,31055.0,31031.0,31029.0,31024.0,31032.0,31060.0,31027.0,31039.0,31037.0,30926.0,30325.0,30482.0,30451.0,30578.0,30811.0,30762.0,30972.0,30716.0,30988.0,30966.0,30969.0,30988.0,30980.0,30966.0,31011.0,31022.0,31012.0,31011.0,31029.0,31055.0,31010.0,30950.0,31000.0,31016.0,31005.0,30985.0,31029.0,31001.0,31001.0,31016.0,31003.0,30981.0,30938.0,30971.0,31012.0,30988.0,31017.0,30990.0,31037.0,31031.0,31024.0,31038.0,31021.0,31032.0,30988.0,31002.0,31043.0,31007.0,31045.0,31022.0,31035.0,31039.0,31035.0,30956.0,30950.0,30988.0,31010.0,31037.0,30981.0,31049.0,31010.0,31049.0,31021.0,31006.0,31026.0,30997.0,30912.0,

In [59]:
logs0.logEvents
    .filterIsInstance(LogEvent.SlotBlockVotes::class.java)
    .filter { it.slotVotes == 0 }
    .toDataFrame()

slot,blockVotes,slotVotes
13151024,0,0
13151025,0,0
13151026,0,0
13151027,0,0
13151028,0,0
13151029,0,0
13151030,0,0
13151031,0,0
13151032,0,0
13151289,0,0
